# Example of usage for **y_call** package: making Y-haplogroup calls #

***

## Description ## 

This Jupter Notebook contains an explained and guided usage of **y_call**, a software designed to make Y-haplogroup calls both on modern and aDNA data. Here, absolute paths in the local clsuter have been considered for input data, but these can be specified by the user at any other location.

***

## Installation ##

Import modules:

In [1]:
# Import modules
import numpy as np
import os  # for saving to folder
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
plt.rcdefaults() # default settings to avoid warnings related to font.
import networkx as nx
from networkx.drawing.nx_pydot import graphviz_layout

import os as os
import sys as sys

from liftover import get_lifter # to convert positions from hg38 to hg19 (used to align reads).
import re # to work with regular expressions.
import pathlib
from pathlib import Path

Import package:

In [20]:
sys.path.append("..")

# Import package from current directory.
from pulldown import y_call

# Change working directory to location of Jupyter Notebook.
notebook_path = os.path.abspath("")
os.chdir(notebook_path)

In [21]:
os.getcwd()

'/mnt/archgen/users/eric_garcia/Jupyter_Vignette'

***

## Understanding necessary input files ##

This section contains a brief explanation on the input files necessary to run the package. Other arguments provided to it stand for filtering options, as well as the creation of a network connecting haplogroups and samples. 

> **bam_list**: relative or abosulte (preferibly) path to a .tsv file containing a list of the routes to different .bam files. This file contains 3 columns in total:
> 1. **iid**: the ID for every sample.
> 2. **bam**: absolute path to each .bam file.
> 3. **sex**: chromosomal sex for every sample. f = female, m = mascle.

In [4]:
bam_list = "/home/eric_garcia_hoyos/y_call/example.tsv"

# Show the content of the file
table1 = pd.read_csv(bam_list, sep="\t", usecols=["iid", "bam", "sex"])
table1.head()

,iid,bam,sex
0,PTN271,/mnt/archgen/Autorun_eager/eager_outputs/SG/PT...,f
1,PTN225,/mnt/archgen/Autorun_eager/eager_outputs/SG/PT...,f
2,PTN209,/mnt/archgen/Autorun_eager/eager_outputs/SG/PT...,m
3,PTN287,/mnt/archgen/Autorun_eager/eager_outputs/SG/PT...,f
4,PTN557,/mnt/archgen/Autorun_eager/eager_outputs/SG/PT...,m


> **path_snps**: relative or abosulte (preferibly) path to a .csv file containing a list of the SNPs considered in the analysis. This file contains 4 columns in total:
> 1. **Coordinates**: genome coordinates in chrY (hg38 version in the example).
> 2. **ANC**: ancestral allele.
> 3. **DER**: derived allele.
> 4. **SNP-ID**: name of the SNP at that location.

In [5]:
path_snps = "/home/eric_garcia_hoyos/y_call/data/input/YBrowse_snps_hg38-runtime.csv"

# Show the content of the file
table2 = pd.read_csv(path_snps, dtype=object, sep=" ", header=None, names = ["Coordinates", "ANC", "DER", "SNP-ID"])
table2.head(10)

,Coordinates,ANC,DER,SNP-ID
0,10000002,A,G,FTD27388
1,10000003,G,T,MF63801
2,10000017,T,A,FTC40137
3,10000024,C,T,TY15024
4,10000025,A,C,BY81325
5,10000032,C,T,TY15023
6,10000044,T,C,MF159896
7,10000054,G,C,FGCLR1744
8,10000056,A,G,FT342891
9,10000062,A,C,Y138007


> **tree**: relative or abosulte (preferibly) path to a .csv file containing child - parent pairs of haplogroups. Although not named in original file, there exist 2 columns:
> 1. **Child**: every haplogroup as a descendant of another haplogroup.
> 2. **Parent**: parental node of the **Child** haplogroup.
> 
> **NOTE**: This file corresponds to the Y-haplogroup tree where samples are placed.

In [6]:
tree = "/home/eric_garcia_hoyos/y_call/data/input/OYchpar.csv"

# Show the content of the file
table3 = pd.read_csv(tree, header=None, names = ["Child", "Parent"])
table3.head(5)

,Child,Parent
0,A-A12220,A00-FGC26001
1,A-A12222,A-FGC28278
2,A-A20843,A-FT185719
3,A-A4995,A-YP2683
4,A-ALK773,A-ALK828


In [7]:
# If Child selected for a specific haplogroup (R-M343):
table3[table3["Child"]=="R-M343"]

,Child,Parent
105132,R-M343,R-PAGES00029


In [8]:
# If Parent selected for a specific haplogroup (R-M343):
table3[table3["Parent"]=="R-M343"]

,Child,Parent
67392,R-BY14355,R-M343
114630,R-YSC0000022,R-M343


> **translation**: relative or abosulte (preferibly) path to a .csv file containing synonymous names for Y-haplogroups to avoid inconsistencies between databases. Although not named in the original file, there exist 2 columns:
> 1. **Large**: nomenclature based on the alphabetical order (ISOGG standard), usually referred as large nomeclature (since names become larger). 
> 2. **Short**: SNP-based nomenclature, where every haplogroup is named after the defining SNP of the branch.

In [9]:
translation = "/home/eric_garcia_hoyos/y_call/data/input/YF-translations.csv"

# Show the content of the file
table4 = pd.read_csv(translation, header=None, names = ["Large", "Short"])[20:30]
table4

,Large,Short
20,A3b2b,A-M118
21,B,B-M60
22,B1,B-M146
23,B2a,B-M150
24,B2a1,B-M109
25,B2a2,B-M108.1
26,B2a2a,B-M43
27,B2b,B-M112
28,B2b1,B-P6
29,B2b2,B-M115


> **mm**: relative or abosulte (preferibly) path to a .tsv file containing the number of samples where a SNP had an ancestral allele. Although not named in the original file, there exist 2 columns:
> 1. **SNP-ID**: name of the SNP.
> 2. **count**: total number of samples with the ancestral allele.

In [10]:
mm = "/home/eric_garcia_hoyos/y_call/data/input/mm_v3.tsv"

# Show the content of the file
table5 = pd.read_csv(mm, sep="\t")
table5.head()

,SNP-ID,count
0,Z9315,115
1,BY184462,103
2,V3167,101
3,FGC27824,89
4,"V6478,Z9327",87


> **branches**: relative or abosulte (preferibly) path to a .csv file containing the name of each haplogroup (first value) and all SNPs contained inside that haplogroup (rest of values).

In [11]:
branches = "/home/eric_garcia_hoyos/y_call/data/input/OYhaps.csv"

# Show the content of the file
with open(branches, "r") as f:
    for line in f.readlines()[4:5]: # read a single line of the document.
        items = line.strip().split() # because every line is: X-XXXX X123 X542 XG56*
        print(f"\nThis is the content of a whole line in the file: {items}")
        if not items:
            continue

        hap = items[0] # only the first value (the haplogroup).
        snps = items[1:] # the rest of values (the SNPs).
        print(f"\n--> Where the haplogroup is: {hap}\n\n--> And the set of SNPs contained inside of it is: {snps}")


This is the content of a whole line in the file: ['A-A4995', 'A4995', 'A4996', 'A4998', 'FTB31618']

--> Where the haplogroup is: A-A4995

--> And the set of SNPs contained inside of it is: ['A4995', 'A4996', 'A4998', 'FTB31618']


***

## Running **y_call** on a single sample ##

So, now that we have all our necessary input files loaded, we can run the program. To do so, we provide a few more parameters so that the analysis considers some aspects:
> **initial**: index for first the sample to analyse.
> 
> **final**: index for the last sample to analyse.
>
> **base_qual**: minimum base quality considered for the call.
>
> **map_qual**: minimum mapping quality for the call.
>
> **reference_genome**: version of the human genome used for the markers' coordinates.
>
> **create_network**: desired network connecting haplogroup and placing samples in it. Y = Yes, N = No.
>
> **width**: width size for the network.
>
> **height**: height size for the network.
>
> **transitions**: filtering for transitions. Y = Yes, N = No.
>
> **ex_limit**: minimum number of samples supporting an ancestral allele for one of the SNPs in mm.

In [12]:
# Run function for the software: y_call.
y_call(bam_list=bam_list, initial=2, final=3,
       base_qual=20, map_qual=25, path_snps=path_snps,
       reference_genome="hg38", create_network="N",
       width=120, height=90, transitions="Y",
       tree=tree, translation=translation, mm=mm,
       branches=branches, ex_limit=5)


## Process input data ##
--------------------
Dataset for filtered markers missing
Creating file...
Loaded 2930035 SNPs
# Out of range positions in hg19: 44
# Positions available: 2879080
# Biallelic SNPs: 2878459
# Ref & Alt different: 2878333
# Ref & Alt ACTG: 2878133
# Unique SNP positions: 2878063
Done
Saved as ./data/output/all_snps.csv
Coresponding .bed file in ./data/output/OY_snps.bed

## Load individuals ##
--------------------
Loaded a total of 1 male individuals.

## Summary coverage statistics for sample KKH019 ##
--------------------
Average Coverage: 0.2952x
Sites covered: 664482 / 2878063
Derived Loci: 2885 / 664482 covered>0

## Process output files for sample KKH019 ##
--------------------
Saved data for pileup as data/output/markers/KKH019/snps_KKH019.tsv
Saved data for pileup (considering only derived sites) as data/output/markers/KKH019/snps_der_KKH019.tsv
Saved data for Y-haplgroup calls as data/output/haplogroups/KKH019/haplogroup_KKH019.tsv
Y-haplogroup simplifi

***

## Understanding the output ##

This section contains a brief explanation on the output files generated by the package. All this information is stored in an automatically created directory, found in ./data/output

> **./data/output/markers/iid/snps_iid.tsv**: this file contains information only for those SNPs covered in the analysis. This file contains a total of 14 columns:
> 1. **SNP-ID**: name of the SNP.
> 2. **chrom**: chromosome where the SNP is located.
> 3. **pos**: genomic coordinates for the SNP.
> 4. **ref**: refference allele.
> 5. **alt**: alternative allele.
> 6. **Y-haplogroup**: name of the Y-haplogroup.
> 7. **Level**: number of steps from Root until current node.
> 8. **[ACGT]**: number of reads supporting each base.
> 9. **ref#**: number of reads supporting the refference allele.
> 10. **alt#**: number of reads supporting the alternative allele.

In [13]:
data1 = pd.read_csv("data/output/markers/KKH019/snps_KKH019.tsv", sep="\t")
data1

,Unnamed: 0,SNP-ID,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
0,0,Y562367,Y,2649888,A,G,NaN,NaN,0,1,0,0,0,1,0
1,1,FTB49845,Y,2649897,C,G,NaN,NaN,0,0,1,0,0,1,0
2,3,Y654946,Y,2649898,T,A,NaN,NaN,0,0,0,0,1,1,0
3,4,MF121600,Y,2649899,A,G,NaN,NaN,0,1,0,0,0,1,0
4,5,MY103,Y,2649911,C,G,NaN,NaN,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498000,713363,BY227657,Y,59033110,A,T,NaN,NaN,0,1,0,0,0,1,0
498001,713364,TY34949,Y,59033135,A,T,NaN,NaN,0,1,0,0,0,1,0
498002,713365,BY227607,Y,59033139,T,C,NaN,NaN,0,0,0,0,1,1,0
498003,713366,Y553748,Y,59033175,T,A,NaN,NaN,0,0,0,0,1,1,0


> **./data/output/markers/iid/snps_der_iid.tsv**: this file follows the same structure as the previous, with the only difference that it contains information only for SNPs considered as derived (alternative allele count higher than ancestral count).  

In [14]:
data2 = pd.read_csv("data/output/markers/KKH019/snps_der_KKH019.tsv", sep="\t")
data2

,Unnamed: 0,SNP-ID,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
0,0,"A10970,E169",Y,13687378,A,T,NaN,NaN,0,0,0,0,9,0,9
1,1,A11362,Y,9816634,C,T,R-A11360,NaN,51,0,0,0,1,0,1
2,2,A12214,Y,3713460,G,A,NaN,NaN,0,1,0,0,0,0,1
3,3,A13510,Y,9771941,T,A,NaN,NaN,0,1,0,0,0,0,1
4,4,A13792,Y,13486797,G,C,NaN,NaN,0,0,3,1,0,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2880,2880,ZS5231,Y,16297466,G,A,NaN,NaN,0,1,0,0,0,0,1
2881,2881,ZS6558,Y,21828241,G,A,NaN,NaN,0,1,0,0,0,0,1
2882,2882,ZS7939,Y,13450812,A,G,NaN,NaN,0,1,0,4,0,1,4
2883,2883,ZS8026,Y,15202045,T,C,NaN,NaN,0,0,1,0,0,0,1


> **./data/output/trees/iid/tree_output_iid.txt**: this file contains information for the whole path until the most derived Y-haplogroup assigned to an individual. Every line is divided as follows:
> 1. **[Haplogroup Name]**: first item in line indicating the name of the Y-haplogroup.
> 2. **Level**: number of steps from Root until current node.
> 3. **DER in branch**: number of SNPs with derived state in current node (marked in grey the total of SNPs for the node).
> 4. **ANC in branch**: number of SNPs with ancestral state in current node (marked in grey the total of SNPs for the node).
> 5. **DER in par.**: number of SNPs with derived state from Root until current node (marked in gray the total of SNPs for parent nodes).
> 6. **ANC in par.**: number of SNPs with ancestral state from Root until current node (marked in gray the total of SNPs for parent nodes).

**NOTE**: if no SNPs for a Y-haplogroup have been covered, the total count is considered 0 and it is labeled as "Not Found". 

In [15]:
with open ("data/output/trees/KKH019/tree_output_KKH019.txt", "r") as file:
    for line in file.readlines():
        print(line,end="")

Tree created for sample KKH019, with most derived Y-haplogroup: I-FTC51616
|
|___> nan, Level: 0, DER in branch: 0, Not Found, ANC in branch: 0, Not Found, DER in par.: 0, Not Found, ANC in par.: 0, Not Found
  |
  |___> A-PR2921, Level: 1, DER in branch: 0, Not Found, ANC in branch: 0, Not Found, DER in par.: 0, Not Found, ANC in par.: 0, Not Found
    |
    |___> A-L1090, Level: 2, DER in branch: 41/41, ANC in branch: 0/41, DER in par.: 0/0, ANC in par.: 0/0
      |
      |___> A-V168, Level: 3, DER in branch: 8/8, ANC in branch: 0/8, DER in par.: 41/41, ANC in par.: 0/41
        |
        |___> A-V221, Level: 4, DER in branch: 0, Not Found, ANC in branch: 0, Not Found, DER in par.: 0, Not Found, ANC in par.: 0, Not Found
          |
          |___> BT-M42, Level: 5, DER in branch: 0, Not Found, ANC in branch: 0, Not Found, DER in par.: 0, Not Found, ANC in par.: 0, Not Found
            |
            |___> CT-M168, Level: 6, DER in branch: 4/4, ANC in branch: 0/4, DER in par.: 49/49

> **./data/output/haplogroups/iid/haplogroup_iid.tsv**: this file contains information supporting each of the Y-haplogroups considered in the analysis. The last row in this table corresponds to the Y-haplogroup assigned to the sample, as it has the highest score. There are a total of 14 different columns:
> 1. **Branch**: name of the Y-haplogroup.
> 2. **Level**: number of steps from Root until current node.
> 3. **Total_SNPs**: number of covered SNPs for the Y-haplogroup.
> 4. **Ancestral**: number of SNPs with ancestral states.
> 5. **Derived**: number of SNPs with derived states.
> 6. **Uncovered**: number of SNPs other than derived or ancestral.
> 7. **#ANC in par.**: number of SNPs with ancestral states in upstream nodes until Root.
> 8. **#DER in par.**: number of SNPs with derived states in upstream nodes until Root.
> 9. **#UNC IN PAR.**: number of SNPs other than derived or ancestral in upstream nodes until Root.
> 10. **Total in par.**: total number of covered SNPs in upstream nodes until Root.
> 11. **Derived/Total**: proportion of derived SNPs and the Total number of SNPs in a Y-haplogroup.
> 12. **#ANC in par./#DER in par.**: ratio between ancestral and derived SNPs i upstream nodes until Root.
> 13. **Support**: number of upstream nodes where Derived/Total proportion is above 0.95.
> 14. **Score**: value obtained from the following calculus:
$$
S = \#\text{DER in par.} + \text{Derived} - 3\left(\#\text{ANC in par.} + \text{Ancestral}\right) - 2\left(\#\text{UNC in par.} + \text{Uncovered}\right)
$$

In [16]:
data4 = pd.read_csv("data/output/haplogroups/KKH019/haplogroup_KKH019.tsv",sep="\t")
data4

,Unnamed: 0,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.,#UNC in par.,Total in par.,Derived/Total,#ANC in par./#DER in par.,Support,Score
0,20515,NaN,0,391648,388742,1303,1603,0,0,0,0,0.003327,NaN,0.0,-1168129.0
1,179,A-ZS601,5,7,7,0,0,298,41,1,340,0.000000,7.268293,1.0,-875.0
2,88,A-L1050,4,131,131,0,0,167,41,1,209,0.000000,4.073171,1.0,-854.0
3,38,A-FT174304,7,1,1,0,0,229,42,1,272,0.000000,5.452381,1.0,-649.0
4,67,A-FTB41632,7,1,1,0,0,229,42,1,272,0.000000,5.452381,1.0,-649.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44366,9500,I-FT72231,36,1,1,0,0,0,107,1,108,0.000000,0.000000,11.0,113.0
44367,11184,I-FTG26286,39,1,1,0,0,0,107,1,108,0.000000,0.000000,11.0,113.0
44368,7105,I-BY166475,37,1,1,0,0,0,107,1,108,0.000000,0.000000,11.0,113.0
44369,12455,I-Y5483,32,1,0,1,0,0,106,1,107,1.000000,0.000000,10.0,115.0


> **./data/output/paths.txt**: this file contains similar information to the tree created above, with the difference that every line is made by the sample ID and the Y-haplogroup names sorted from Root until most derived haplogroup.

In [17]:
with open ("data/output/paths.txt", "r") as file:
    for line in file.readlines():
        print(line,end="")

KKH019: nan,A-PR2921,A-L1090,A-V168,A-V221,BT-M42,CT-M168,CF-P143,F-M89,GHIJK-YSC0001299,HIJK-M578,IJK-L15,IJ-P124,I-YSC0000285,I-M170,I-FGC2430,I-L840,I-M253,I-DF29,I-Y2592,I-Z2336,I-Z2337,I-S6346,I-L22,I-Z2338,I-CTS6868,I-Z74,I-FGC9478,I-S436,I-Y5476,I-Y5478,I-Y5485,I-Y5483,I-FGC9455,I-FGC9453,I-BY3428,I-FT14651,I-FTC51616


> **./data/output/scores.csv**: this is the file containing information for the most derived Y-haplogroup assigned to every sample considered in the study. Although not named in original file, there exist 5 columns:
> 1. **Sample_ID**: name for the sample analysed.
> 2. **Y-haplogroup**: name of the Y-haplogroup assigned to the sample.
> 3. **Level**: number of steps from Root until current node.
> 4. **#ANC in par./#DER in par.**: ratio between the number of ancestral SNPs and derived SNPs in parent nodes of the Y-haplogroup assigned.
> 5. **Score**: value obtained for the calculation of the pre-defined score.

In [18]:
data3 = pd.read_csv("data/output/scores.csv", header=None, names = ["Sample_ID", "Y-haplogroup", "Level", "#ANC in par./#DER in par.", "Score"])
data3

,Sample_ID,Y-haplogroup,Level,#ANC in par./#DER in par.,Score
0,KKH019,I-FTC51616,37,0.0,117


***

## Running **y_call** on the whole set of samples ##

In [23]:
# Run function for the software: y_call.
y_call(bam_list=bam_list, initial=0, final=0,
       base_qual=20, map_qual=25, path_snps=path_snps,
       reference_genome="hg38", create_network="Y",
       width=120, height=90, transitions="Y",
       tree=tree, translation=translation, mm=mm,
       branches=branches, ex_limit=5)


## Process input data ##
--------------------
Dataset for filtered markers already saved: ./data/output/all_snps.csv
Corresponding .bed file in data/output/OY_snps.bed

## Load individuals ##
--------------------
Loaded a total of 4 male individuals.

## Summary coverage statistics for sample PTN209 ##
--------------------
Average Coverage: 7.3611x
Sites covered: 2378348 / 2878063
Derived Loci: 6719 / 2378348 covered>0

## Process output files for sample PTN209 ##
--------------------
Saved data for pileup as data/output/markers/PTN209/snps_PTN209.tsv
Saved data for pileup (considering only derived sites) as data/output/markers/PTN209/snps_der_PTN209.tsv
Saved data for Y-haplgroup calls as data/output/haplogroups/PTN209/haplogroup_PTN209.tsv
Y-haplogroup simplified tree saved as data/output/trees/PTN209/tree_output_PTN209.txt

 ## Update global output files ##
--------------------
Path to most derived haplogroup included in data/output/paths.txt

## Summary coverage statistics for sam